In [1]:
import numpy as np
import pandas as pd

In [2]:
dataset = pd.read_csv("../Datasets/FinalCropYield.csv")

In [3]:
dataset.head(2)

,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0,321.0
1,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.0,641.0


In [4]:
dataset = dataset.drop(columns=["Crop_Year"])


In [5]:
dataset.head(2)

,State_Name,District_Name,Season,Crop,Area,Production
0,Andaman and Nicobar Islands,NICOBARS,Kharif,Rice,102.0,321.0
1,Andaman and Nicobar Islands,NICOBARS,Whole Year,Banana,176.0,641.0


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

In [7]:
dataset = pd.read_csv("../Datasets/crop_production.csv")
dataset = dataset.dropna()

dataset["Yield"] = dataset["Production"] / dataset["Area"]
dataset = dataset.drop(columns=["Production"])

X = dataset.drop(columns=["Yield"])
y = dataset["Yield"]

categorical_cols = ["State_Name", "District_Name", "Season", "Crop"]
numeric_cols = ["Area", "Crop_Year"]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

In [8]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("regressor", XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        random_state=42
    ))
])

In [10]:
model.fit(X_train, y_train)


,steps,"[('preprocessing', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [11]:
print("R2 Score:", model.score(X_test, y_test))


R2 Score: 0.8798577727493659


In [12]:
import pickle

with open("CropYieldPipeline.pkl", "wb") as f:
    pickle.dump(model, f)
